<a href="https://colab.research.google.com/github/tkratt/GARCH_vs_LSTM_OPTION_PRICING/blob/main/FIGARCH_with_AAPL_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FIGARCH(1,d,1) Model

In [2]:
!pip install arch

import numpy as np
import pandas as pd

from arch.univariate import (
    ConstantMean,
    FIGARCH,
    Normal
)

def figarch_model(data):

    returns = data.dropna()

    model = ConstantMean(returns)

    model.volatility = FIGARCH(
        p=1,
        q=1
    )

    model.distribution = Normal()

    result = model.fit(disp="off")

    forecast = result.forecast(
        horizon=1
    )

    variance_1d = forecast.variance.iloc[-1, 0]

    daily_vol = np.sqrt(
        variance_1d
    )

    forecast_date = (
        returns.index[-1]
        + pd.offsets.BDay(1)
    )

    print("Parameters")
    print(result.params)

    print("\nLogLikelihood")
    print(result.loglikelihood)

    print("\nAIC")
    print(result.aic)

    print("\nBIC")
    print(result.bic)

    print("\nForecast Date")
    print(forecast_date)

    print("\n1-Day Volatility")
    print(f"{daily_vol:.4f}%")

    return result

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 982.9/982.9 kB 9.3 MB/s eta 0:00:00


In [3]:
import yfinance as yf

aapl = yf.download(
    "AAPL",
    start="2015-01-01",
    auto_adjust=True,
    progress=False
)

aapl["Log_Return"] = (
    np.log(
        aapl["Close"] /
        aapl["Close"].shift(1)
    )
) * 100

returns = aapl["Log_Return"].dropna()

figarch_result = figarch_model(returns)

Parameters
mu       0.138569
omega    0.183805
phi      0.296819
d        0.338236
beta     0.508727
Name: params, dtype: float64

LogLikelihood
-5650.074362478996

AIC
11310.148724957991

BIC
11340.09143987168

Forecast Date
2026-09-24 00:00:00

1-Day Volatility
1.4086%
